In [ ]:
import pandas as pd
import numpy as np
import h3
import matplotlib.pyplot as plt
import geopandas as gpd
import folium
import seaborn as sns

This Notebook is for descriptive temporal analysis only. As we do not need Census Tract values for this, we can work with the big dataset of the taxidata.

In [ ]:
taxi_data = pd.read_parquet(
    "../data/processed/taxi_data_processed_big.parquet"
)
taxi_data.info()

## 1. Preparation


In [ ]:
#drop all spatial columns as they are not needed for temporal analysis
taxi_data = taxi_data.drop(columns=["Pickup Census Tract", "Dropoff Census Tract", 
                                    "Pickup Community Area", "Dropoff Community Area", 
                                    "h3_index_pickup_7", "h3_index_dropoff_7",
                                    "h3_index_pickup_8", "h3_index_dropoff_8",
                                    "h3_index_pickup_9", "h3_index_dropoff_9",
                                    "Pickup Centroid Latitude", "Dropoff Centroid Latitude",
                                    "Pickup Centroid Longitude", "Dropoff Centroid Longitude"])

In [ ]:
#convert to datetime
taxi_data["Trip Start Timestamp"] = pd.to_datetime(
    taxi_data["Trip Start Timestamp"]
)

taxi_data["Trip End Timestamp"] = pd.to_datetime(
    taxi_data["Trip End Timestamp"]
)

We create further features to analyse the dataset.

In [ ]:
#create temporal features
taxi_data["date"] = taxi_data["Trip Start Timestamp"].dt.date

taxi_data["year"] = taxi_data["Trip Start Timestamp"].dt.year

taxi_data["month"] = taxi_data["Trip Start Timestamp"].dt.month

taxi_data["week"] = taxi_data["Trip Start Timestamp"].dt.isocalendar().week
taxi_data["weekday_name"] = taxi_data["Trip Start Timestamp"].dt.day_name()
taxi_data["is_weekend"] = taxi_data["weekday_name"].isin(["Saturday", "Sunday"])

taxi_data["hour"] = taxi_data["Trip Start Timestamp"].dt.hour

#trip duration in minutes
taxi_data["Trip Minutes"] = taxi_data["Trip Seconds"] / 60

#average speed (mph)
taxi_data["average_speed"] = np.where(
    taxi_data["Trip Seconds"] > 0,
    taxi_data["Trip Miles"] / (taxi_data["Trip Seconds"] / 3600),
    np.nan,
)

taxi_data

## 2. Overall Demand
### 2.1 Daily Demand

In [ ]:
# Number of trips per day
daily_demand = (
    taxi_data
    .groupby("date")
    .size()
    .rename("trip_count")
    .reset_index()
)

daily_demand

In [ ]:
plt.figure(figsize=(14,5))

plt.plot(
    daily_demand["date"],
    daily_demand["trip_count"],
    linewidth=1
)

plt.title("Daily Taxi Demand")
plt.xlabel("Date")
plt.ylabel("Number of Trips")

plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

The daily number of taxi trips exhibits a pronounced weekly cycle. Demand regularly increases and decreases at approximately seven-day intervals, indicating systematic differences between weekdays and weekends.

Beyond this weekly pattern, the overall demand shows moderate long-term fluctuations. Demand is generally lower at the beginning of 2024, increases towards mid-2024, decreases again around the turn of the year 2024/2025, and rises afterwards.

Several days exhibit unusually low trip counts. These may correspond to public holidays, extreme weather events, data collection issues, or incomplete observations. The final observation with zero trips is most likely caused by an incomplete recording of the last day and should therefore not be interpreted as an actual demand pattern.

### 2.2 Weekly Demand

In [ ]:
weekly_demand = (
    taxi_data
    .groupby(["year", "week"])
    .size()
    .rename("trip_count")
    .reset_index()
)

weekly_demand["year_week"] = (
    weekly_demand["year"].astype(str)
    + "-W"
    + weekly_demand["week"].astype(str).str.zfill(2)
)

weekly_demand

In [ ]:
#TODO: delete data from last day from the dataset, as it shows an uncomplete pattern and could distort the analysis

In [ ]:
# Position of the first week of each year
year_start_positions = (
    weekly_demand
    .groupby("year")
    .head(1)
    .index
)

year_labels = (
    weekly_demand
    .groupby("year")
    .head(1)["year"]
)

plt.figure(figsize=(14,5))

plt.plot(
    weekly_demand.index,
    weekly_demand["trip_count"],
    linewidth=2
)

# Mark the beginning of each year
for pos in year_start_positions:
    plt.axvline(pos, color="grey", linestyle="--", alpha=0.5)

plt.xticks(
    year_start_positions,
    year_labels
)

plt.title("Weekly Taxi Demand")
plt.xlabel("Year")
plt.ylabel("Number of Trips")

plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

Aggregating demand on a weekly basis removes most of the short-term fluctuations and highlights broader temporal trends. Weekly demand varies between approximately 80,000 and 145,000 trips.

Demand decreases noticeably around the beginning of each year before recovering in the following months. Such recurring reductions are likely related to seasonal effects, for example lower travel activity during holiday periods.

The sharp decline in the final week is caused by incomplete data and should therefore be excluded from further analyses.

## 3 Intraday patterns

### 3.1 Demand by Hour of the Day

In [ ]:
hourly_demand = taxi_data.groupby("hour").size()
weekly_demand = taxi_data.groupby("weekday").size()

#hourly_demand

In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(hourly_demand.index, hourly_demand.values)

plt.title("Demand by Hour of Day")
plt.xlabel("Hour of Day")
plt.ylabel("Number of Trips")
plt.xticks(range(24))

plt.show()

The observed pattern reflects typical daily mobility behaviour. Demand is lowest during the early morning hours when most people are inactive. Beginning with the morning commute, demand increases rapidly and remains high throughout normal business hours. The peak in the late afternoon is likely associated with commuters returning home as well as increased leisure activities after work.

### 3.2 Weekday vs Weekend

In [ ]:
hourly_weekend = (
    taxi_data.groupby(["date", "hour", "is_weekend"])
    .size()
    .reset_index(name="trips")
)

hourly_weekend_avg = (
    hourly_weekend
    .groupby(["hour", "is_weekend"])["trips"]
    .mean()
    .unstack()
)

In [ ]:
hours = hourly_weekend_avg.index

x = np.arange(len(hours))
width = 0.4

plt.figure(figsize=(8,4))

plt.bar(
    x - width/2,
    hourly_weekend_avg[False],
    width,
    label="Weekday"
)

plt.bar(
    x + width/2,
    hourly_weekend_avg[True],
    width,
    label="Weekend"
)

plt.xticks(x, hours)

plt.xlabel("Hour of Day")
plt.ylabel("Number of Trips")
plt.title("Taxi Demand by Hour: Weekday vs Weekend")

plt.legend()
plt.grid(axis="y", alpha=0.3)

plt.show()

Weekdays exhibit pronounced morning and evening peaks, indicating commuting behaviour during regular working hours. In contrast, weekends show substantially higher demand during the night and early morning, suggesting increased leisure and nightlife-related travel. The absence of a strong morning peak further reflects the reduced importance of work-related trips on weekends.

In [ ]:
# Average hourly demand for each weekday
heatmap_data = (
    taxi_data
    .groupby(["weekday", "hour"])
    .size()
    .groupby(level=[0, 1])
    .mean()
    .unstack(fill_value=0)
)

plt.figure(figsize=(14, 5))

sns.heatmap(
    heatmap_data,
    cmap="YlOrRd",
    linewidths=0.5,
    cbar_kws={"label": "Average Number of Trips"}
)

plt.title("Average Hourly Taxi Demand by Weekday")
plt.xlabel("Hour of Day")
plt.ylabel("Weekday")

plt.tight_layout()
plt.show()

The heatmap highlights highly regular intraday demand patterns across all weekdays. Demand is consistently lowest during the early morning hours and increases rapidly after approximately 6 a.m. From Tuesday to Thursday, demand remains at a particularly high level throughout the working day, reaching its maximum during the late afternoon. In contrast, Saturdays and Sundays exhibit a noticeably weaker morning peak and a flatter demand profile, reflecting reduced commuter traffic and increased leisure-oriented travel.

In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(weekly_demand.index, weekly_demand.values)

plt.title("Taxi Demand by Day of the week")
plt.xlabel("Day of the week")
plt.ylabel("Number of Trips")

plt.show()

trip length analysis

In [ ]:
taxi_data["Trip Miles"].describe()

In [ ]:
plt.figure(figsize=(8,4))
plt.hist(
    taxi_data["Trip Miles"],
    bins=100
)
plt.xlabel("Trip Length (Miles)")
plt.ylabel("Number of Trips (log)")
plt.title("Distribution of Trip Lengths")
plt.yscale("log")
plt.show()

In [ ]:
hourly_trip_length = (
    taxi_data
    .groupby(
        taxi_data["hour"])["Trip Miles"]
    .mean()
)

In [ ]:
plt.figure(figsize=(8,4))
plt.plot(hourly_trip_length, marker = "o")
plt.xlabel("hour")
plt.ylabel("Average Trip Length (Miles)")
plt.title("Average hourly Trip Lengths")
plt.show()

price analysis

In [ ]:
plt.figure(figsize=(8,4))

plt.hist(
    taxi_data["Trip Total"],
    bins=50
)

plt.title("Distribution of Trip Prices")
plt.xlabel("Trip Price ($)")
plt.ylabel("Number of Trips")

plt.show()

In [ ]:
fare_99 = taxi_data["Trip Total"].quantile(0.99)

plt.figure(figsize=(8,4))

plt.hist(
    taxi_data.loc[
        taxi_data["Trip Total"] <= fare_99,
        "Trip Total"
    ],
    bins=50
)

plt.title("Distribution of Trip Prices")
plt.xlabel("Trip Price ($)")
plt.ylabel("Number of Trips")

plt.show()

In [ ]:
taxi_data["Trip Total"].describe(
    percentiles=[0.5,0.9,0.95,0.99]
)

In [ ]:
hourly_avg_price = (
    taxi_data
    .groupby("hour")["Trip Total"]
    .mean()
)

In [ ]:
plt.figure(figsize=(8,4))

plt.plot(
    hourly_avg_price.index,
    hourly_avg_price.values,
    marker="o"
)

plt.title("Average Trip Price by Hour")
plt.xlabel("Hour of Day")
plt.ylabel("Average Price ($)")
plt.xticks(range(24))
plt.grid(alpha=0.3)

plt.show()